# GNN Benchmarking on the Wollombi Flood Dataset

Benchmarking several Graph Neural Network architectures for flood prediction, using the
[DUALFloodGNN](https://github.com/acostacos/dual_flood_gnn) codebase and the Wollombi
HEC-RAS flood dataset.

**Models compared:** GCN, GAT, GIN, GINE, GraphSAGE (node-only baselines) and
DUALFloodGNN (node volume + edge flow, physics-informed).

**Task:** predict water volume at mesh nodes (and water flow at edges for DUALFloodGNN),
evaluated on 4 held-out flood events. Metrics: RMSE, MAE, NSE, CSI.

**Environment:** Kaggle, GPU T4. Trained 50 epochs per model, seed 42.

---
### Setup notes
- **Dataset** is not in the GitHub repo — it is downloaded separately from the
  [USYD library](https://ses.library.usyd.edu.au/handle/2123/35293) and attached to this
  notebook as Kaggle datasets (HDF simulation files, GEOMETRY shapefiles, train/test CSVs).
- The dataset ships raw shapefiles (`updated_cell_centers.shp`, `links.shp`) whose columns
  already match what the loader needs; they are copied to the filenames the config expects.
- One bug in the upstream `train.py` (a stale variable name in the autoregressive test path)
  is patched below.

## 1. Environment check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Clone the DUALFloodGNN repository

In [ ]:
%cd /kaggle/working
!git clone https://github.com/acostacos/dual_flood_gnn.git
%cd dual_flood_gnn

## 3. Install dependencies

`torch_geometric` core is pure-Python and installs against any recent PyTorch. The compiled PyG extras (`pyg_lib`, `torch_scatter`, ...) have no wheels for the Kaggle PyTorch build and are not needed for the standard models, so they are skipped.

In [ ]:
!pip install torch_geometric -q
print("torch_geometric installed (compiled extras skipped — not required for these models)")

In [ ]:
# Verify the GNN layers import
import torch_geometric
from torch_geometric.nn import GCNConv, GATConv, GINEConv, SAGEConv
print("torch_geometric", torch_geometric.__version__, "- core layers OK")

### 3a. Install the repo's other requirements

The upstream `requirements.txt` is UTF-16 encoded and pins the compiled PyG extras; this reads it robustly and installs only the packages this environment needs.

In [ ]:
# requirements.txt is UTF-16 encoded (BOM). Read it accordingly and drop packages
# that either don't have wheels here or are already satisfied by Kaggle.
raw = open('requirements.txt', 'rb').read()
if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
    text = raw.decode('utf-16')
else:
    text = raw.decode('utf-8', errors='ignore')

skip = ('pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster',
        'torch_spline_conv', 'torch_geometric', 'torch', 'numpy')
clean = [l.strip() for l in text.splitlines()
         if l.strip() and not l.strip().startswith('#')
         and not any(l.strip().lower().startswith(s) for s in skip)]

with open('requirements_clean.txt', 'w') as f:
    f.write('\n'.join(clean) + '\n')
print("Installing:", clean)
!pip install -r requirements_clean.txt -q

In [ ]:
import h5py, geopandas, rasterio, yaml, pandas
print("Data libraries OK")

## 4. Prepare the dataset

The attached Kaggle datasets are placed into the folder layout the repo's data loader
expects (`data/datasets/raw/`), using the exact folder and file names it looks for:
- `HEC-RAS Results/` — the HDF simulation files
- `GEOMETRY/` — DEM + shapefiles (copied, so we can add the expected filenames)
- `train.csv`, `test.csv`

In [ ]:
import os, glob, shutil

RAW = "/kaggle/working/dual_flood_gnn/data/datasets/raw"
GEO = os.path.join(RAW, "GEOMETRY")
os.makedirs(RAW, exist_ok=True)

# Locate the source folders/files across all attached Kaggle datasets
geo_src   = os.path.dirname(glob.glob("/kaggle/input/**/DEM.tif", recursive=True)[0])
hdf_src   = os.path.dirname(glob.glob("/kaggle/input/**/Model_01.p22.hdf", recursive=True)[0])
train_csv = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
test_csv  = glob.glob("/kaggle/input/**/test*.csv", recursive=True)[0]  # handles "test (1).csv"

# GEOMETRY: copy into a writable folder (~15 MB) so we can add expected filenames
if os.path.islink(GEO) or os.path.exists(GEO):
    shutil.rmtree(GEO, ignore_errors=True)
    if os.path.islink(GEO): os.remove(GEO)
shutil.copytree(geo_src, GEO, dirs_exist_ok=True)

# HDF files (~4.5 GB): symlink under the exact name train.csv references
hdf_dst = os.path.join(RAW, "HEC-RAS Results")
if os.path.islink(hdf_dst) or os.path.exists(hdf_dst): os.remove(hdf_dst)
os.symlink(hdf_src, hdf_dst)

# CSVs: copy in with the names the config expects
shutil.copy(train_csv, os.path.join(RAW, "train.csv"))
shutil.copy(test_csv,  os.path.join(RAW, "test.csv"))

print("HDF files reachable:", os.path.exists(os.path.join(hdf_dst, "Model_01.p22.hdf")))
print("raw/ contents:", sorted(os.listdir(RAW)))

### 4a. Match shapefile names to the config

The config references `cell_centers_with_ele.shp` and `links_with_slope.shp`. The raw
dataset provides the same geometry (with all required columns: `X`, `Y`, `Elevation1` for
nodes; `from_node`, `to_node`, `length`, `slope` for edges) under the names
`updated_cell_centers.shp` and `links.shp`. We copy each shapefile component to the
expected name.

In [ ]:
mappings = [
    ("updated_cell_centers", "cell_centers_with_ele"),
    ("links", "links_with_slope"),
]
for src_base, dst_base in mappings:
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        src = os.path.join(GEO, src_base + ext)
        dst = os.path.join(GEO, dst_base + ext)
        if os.path.exists(src):
            shutil.copy(src, dst)
print("Shapefiles renamed to match config. GEOMETRY now contains:")
for f in sorted(os.listdir(GEO)):
    print("  ", f)

## 5. Patch an upstream bug

In `train.py`, the autoregressive test path calls `del dataset`, but under autoregressive
training the datasets are named `train_dataset` / `val_dataset`, so the original line raises
`UnboundLocalError`. This patch fixes the variable name.

In [ ]:
with open("train.py") as f:
    content = f.read()

content = content.replace(
    "        # Clear memory before loading test dataset\n        del dataset\n        gc.collect()",
    "        # Clear memory before loading test dataset\n        del train_dataset, val_dataset\n        gc.collect()"
)
with open("train.py", "w") as f:
    f.write(content)
print("Patched train.py (autoregressive test-path variable name).")

## 6. Set training length

Set both configs to 50 epochs for this benchmarking pass. (The upstream defaults are 300-600; 50 gives a fast, consistent comparison across models. For final/converged numbers, increase this.)

In [ ]:
import re
for cfg in ["configs/standard_gnn_config.yaml", "configs/config.yaml"]:
    with open(cfg) as f:
        lines = f.readlines()
    for i, l in enumerate(lines):
        if l.strip().startswith("num_epochs") and not l.strip().startswith("num_epochs_dyn_loss"):
            lines[i] = re.sub(r":\s*\d+", ": 50", l)
    with open(cfg, "w") as f:
        f.writelines(lines)
print("Both configs set to 50 epochs")

## 7. Train and evaluate all models

Node-only baselines use `standard_gnn_config.yaml`; the physics-informed DUALFloodGNN uses
`config.yaml` (mass-conservation losses enabled). Each model trains from a fresh random
initialisation (no checkpoint) and is tested on the 4 held-out events.

In [ ]:
import subprocess

standard_models = ["GCN", "GAT", "GIN", "GINE", "GraphSAGE"]
physics_models  = ["DUALFloodGNN"]

def run(model, cfg):
    print(f"\n{'='*60}\nTRAINING {model}  (config: {cfg})\n{'='*60}", flush=True)
    r = subprocess.run(
        ["python", "train.py", "--config", f"configs/{cfg}",
         "--model", model, "--with_test", "True", "--seed", "42", "--device", "cuda"]
    )
    print(f"{model} finished with exit code {r.returncode}", flush=True)

for m in standard_models:
    run(m, "standard_gnn_config.yaml")
for m in physics_models:
    run(m, "config.yaml")

print("\nALL MODELS DONE")

## 8. Results table

Aggregate the per-event test metrics into one comparison table (averaged across the 4 test events).

In [ ]:
import numpy as np, glob, os, re
import pandas as pd

rows = []
for f in sorted(glob.glob("saved_metrics/*_test_metrics.npz")):
    d = np.load(f, allow_pickle=True)
    name = os.path.basename(f)
    model = name.split("_")[0]
    runid = re.search(r"runid_(\d+)", name)
    row = {"model": model, "run_id": runid.group(1) if runid else "?"}
    for k in d.files:
        arr = d[k]
        try:
            row[k] = float(arr) if arr.size == 1 else float(np.mean(arr.astype(float)))
        except Exception:
            pass
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby("model").mean(numeric_only=True).round(3)
cols = [c for c in ["rmse", "mae", "nse", "csi"] if c in summary.columns]
print(summary[cols] if cols else summary)
summary.to_csv("model_comparison.csv")
print("\nSaved model_comparison.csv")

## 9. Plots

### 9a. Training loss and validation RMSE per epoch

In [ ]:
import numpy as np, glob, os
import matplotlib.pyplot as plt

tfiles = sorted(glob.glob("training_stats/*_train_stats.npz"))
seen = {}
for f in tfiles:
    seen.setdefault(os.path.basename(f).split("_")[0], f)

order  = ["GINE","GIN","GCN","DUALFloodGNN","GraphSAGE","GAT"]
colors = {"GINE":"#1f9e89","GIN":"#3b6ea5","GCN":"#9467bd",
          "DUALFloodGNN":"#e07b39","GraphSAGE":"#8c564b","GAT":"#d62728"}

for key, ylabel, title, fname in [
    ("val_node_rmse", "Validation Node RMSE (log)", "Validation RMSE vs Epoch", "val_rmse.png"),
    ("train_epoch_loss", "Training Loss (MSE, log)", "Training Loss vs Epoch", "train_loss.png"),
]:
    plt.figure(figsize=(11,6))
    for m in order:
        if m in seen:
            y = np.array(np.load(seen[m], allow_pickle=True)[key], float)
            plt.plot(range(1,len(y)+1), y, marker='o', ms=3, lw=1.2, label=m, color=colors[m])
    plt.xlabel("Epoch"); plt.ylabel(ylabel); plt.title(title + " - GNN Models (50 epochs)")
    plt.yscale("log"); plt.legend(title="Model"); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()

### 9b. Predicted vs. true water volume (all models)

In [ ]:
from collections import defaultdict

mfiles = sorted(glob.glob("saved_metrics/*_test_metrics.npz"))
by_model = defaultdict(list)
for f in mfiles:
    by_model[os.path.basename(f).split("_")[0]].append(f)

fig, axes = plt.subplots(2, 3, figsize=(15,10)); axes = axes.flatten()
for i, m in enumerate(order):
    ax = axes[i]
    if m not in by_model: ax.axis("off"); continue
    P, T = [], []
    for f in by_model[m]:
        d = np.load(f, allow_pickle=True)
        P.append(np.array(d["pred"], float).ravel()); T.append(np.array(d["target"], float).ravel())
    p, t = np.concatenate(P), np.concatenate(T)
    if p.size > 30000:
        idx = np.random.choice(p.size, 30000, replace=False); p, t = p[idx], t[idx]
    ax.scatter(t, p, s=3, alpha=0.25, color="#3b6ea5")
    lo, hi = min(t.min(),p.min()), max(t.max(),p.max())
    ax.plot([lo,hi],[lo,hi],"r--",lw=1.5,label="Perfect (y=x)")
    ax.set_title(m); ax.set_xlabel("True water volume"); ax.set_ylabel("Predicted water volume")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle("Predicted vs. True Water Volume - Test Set (50 epochs)", fontsize=14)
plt.tight_layout(); plt.savefig("scatter_pred_vs_true.png", dpi=150, bbox_inches="tight"); plt.show()

### 9c. DUALFloodGNN dual objectives — node volume and edge flow

In [ ]:
seen_n = set(); NP=[]; NT=[]; EP=[]; ET=[]
for f in by_model.get("DUALFloodGNN", []):
    d = np.load(f, allow_pickle=True)
    NP.append(np.array(d["pred"], float).ravel());       NT.append(np.array(d["target"], float).ravel())
    EP.append(np.array(d["edge_pred"], float).ravel());  ET.append(np.array(d["edge_target"], float).ravel())

if NP:
    np_, nt_ = np.concatenate(NP), np.concatenate(NT)
    ep_, et_ = np.concatenate(EP), np.concatenate(ET)
    def sub(a,b,n=30000):
        if a.size>n:
            i=np.random.choice(a.size,n,replace=False); return a[i],b[i]
        return a,b
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,6))
    for ax, (x,y), title, c in [(a1, sub(nt_,np_), "Node Water Volume", "#3b6ea5"),
                                 (a2, sub(et_,ep_), "Edge Water Flow", "#1f9e89")]:
        ax.scatter(x,y,s=3,alpha=0.25,color=c)
        lo,hi=min(x.min(),y.min()),max(x.max(),y.max())
        ax.plot([lo,hi],[lo,hi],"r--",lw=1.5,label="Perfect (y=x)")
        ax.set_title(title); ax.set_xlabel("True"); ax.set_ylabel("Predicted")
        ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
    plt.suptitle("DUALFloodGNN: Dual Objectives - Volume (nodes) & Flow (edges)", fontsize=14)
    plt.tight_layout(); plt.savefig("dualflood_volume_and_flow.png", dpi=150, bbox_inches="tight"); plt.show()

## 10. Summary of findings

- **GINE** is the strongest baseline (best RMSE, least-negative NSE, best CSI) — it is the
  only baseline using edge features (slope, length), which physically govern water flow.
- **GAT** diverges numerically at the default learning rate (predictions ~10^10) — a
  stability issue, addressable with a lower learning rate / gradient clipping.
- **DUALFloodGNN** is undertrained at 50 epochs; its harder dual objective (node volume +
  edge flow under mass conservation) needs the full-length run to converge.
- All NSE values are negative at 50 epochs, i.e. no model yet beats a mean baseline on water
  magnitude — expected for this short training length. The model *ranking* is the reliable
  takeaway; converged numbers require 300-600 epochs.

**Next steps:** full-length training with early stopping, GAT learning-rate tuning, remaining
edge/physics model variants, and multiple seeds for statistical reliability.